# 06 - Random Forest Model  
Hotel Booking Demand (Cancellation Prediction)

## Notebook purpose
- Train and tune a Random Forest classifier
- Save all required Member 4 artifacts for report and comparison
- Keep output structure consistent with other model notebooks

## Working directory setup

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    repo_root = subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"],
        text=True,
    ).strip()
    os.chdir(repo_root)
except Exception:
    pass

cwd = Path.cwd()
if str(cwd) not in sys.path:
    sys.path.append(str(cwd))

print("Working directory:", cwd)

## Imports and artifact paths

In [ ]:
import json
import pandas as pd

from src.member4_rf import run_member4_random_forest
from src.config import DEFAULT_DATA_PATH

ART = {
    "models": Path("artifacts/models"),
    "metrics": Path("artifacts/metrics"),
    "plots": Path("artifacts/plots"),
    "reports": Path("artifacts/reports"),
}
for p in ART.values():
    p.mkdir(parents=True, exist_ok=True)

print("Default data path:", DEFAULT_DATA_PATH)

## Dataset check

In [ ]:
data_path = Path(DEFAULT_DATA_PATH)
print("Dataset exists:", data_path.exists(), "->", data_path)
if not data_path.exists():
    raise FileNotFoundError("Dataset not found. Put hotel_bookings.csv in data/raw/.")

## Train and tune Random Forest

In [ ]:
# For slower machines, reduce max_tune_rows (e.g., 25000)
results = run_member4_random_forest(
    data_path=str(data_path),
    scoring="f1",
    max_tune_rows=40000,
)
results

## Load and display saved metrics

In [ ]:
with open(ART["metrics"] / "rf_test_metrics.json", "r", encoding="utf-8") as f:
    rf_metrics = json.load(f)

with open(ART["metrics"] / "rf_best_params.json", "r", encoding="utf-8") as f:
    rf_best_params = json.load(f)

print("Best params:")
print(rf_best_params)

metric_order = [
    "accuracy", "balanced_accuracy", "precision", "recall",
    "f1", "roc_auc", "pr_auc", "log_loss"
]
rf_table = pd.DataFrame([
    {"model": "random_forest", **{k: rf_metrics.get(k) for k in metric_order}}
])
rf_table

## Feature importance (Top 15)

In [ ]:
fi_path = ART["metrics"] / "rf_feature_importance.csv"
if fi_path.exists():
    fi_top15 = pd.read_csv(fi_path).sort_values("importance", ascending=False).head(15)
    fi_top15
else:
    print("Feature importance file not found:", fi_path)

## Artifact verification checklist

In [ ]:
expected = [
    ART["models"] / "rf_pipeline.joblib",
    ART["metrics"] / "rf_cv_results.csv",
    ART["metrics"] / "rf_best_params.json",
    ART["metrics"] / "rf_test_metrics.json",
    ART["metrics"] / "rf_feature_importance.csv",
    ART["plots"] / "rf_confusion_matrix.png",
    ART["plots"] / "rf_roc_curve.png",
    ART["plots"] / "rf_pr_curve.png",
    ART["plots"] / "rf_feature_importance.png",
    ART["reports"] / "rf_classification_report.txt",
    ART["reports"] / "rf_notes.md",
]

check_df = pd.DataFrame({
    "artifact": [str(p) for p in expected],
    "exists": [p.exists() for p in expected],
})
check_df

## Results discussion (for report)
- Random Forest captures non-linear interactions in booking behavior.
- Hyperparameter tuning is done with GridSearchCV using F1.
- Results include confusion matrix, ROC/PR curves, and feature importance.
- Use the saved `rf_test_metrics.json` directly in `07_model_comparison.ipynb`.